<a href="https://colab.research.google.com/github/dhana0020/Deep-Learning/blob/main/RNN_sentiment_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter
import re

In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

In [ ]:
counter = Counter()

for sample in train_data:
    text = clean_text(sample["text"])
    counter.update(text.split())

vocab_size = 10000
most_common = counter.most_common(vocab_size - 2)

vocab = {"<PAD>": 0, "<UNK>": 1}

for idx, (word, _) in enumerate(most_common, start=2):
    vocab[word] = idx

In [ ]:
def encode(text, max_len):
    text = clean_text(text)
    tokens = text.split()
    encoded = [vocab.get(word, 1) for word in tokens]

    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]

    return encoded

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, data, max_len):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        encoded = encode(text, self.max_len)

        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.float)

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            nonlinearity="relu"
        )

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
def train_and_evaluate(MAX_LEN):

    print("\nRunning for MAX_LEN =", MAX_LEN)

    train_dataset = IMDBDataset(train_data, MAX_LEN)
    test_dataset = IMDBDataset(test_data, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    model = SentimentRNN(len(vocab))
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

    # ---- Training ----
    num_epochs = 5

    for epoch in range(num_epochs):
        model.train()
        correct = 0
        total = 0

        for texts, labels in train_loader:
            optimizer.zero_grad()

            outputs = model(texts).view(-1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        print(f"Epoch {epoch+1} | Train Acc: {correct/total:.4f}")

    # ---- Testing ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for texts, labels in test_loader:
            outputs = model(texts).view(-1)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total
    print("Test Accuracy:", round(test_acc, 4))

    return  model, test_acc

In [ ]:
results = {}



for length in [80, 100, 120]:
    model, acc = train_and_evaluate(length)
    results[length] = acc
print("\nFinal Comparison:")
for k, v in results.items():
    print(f"MAX_LEN = {k} → Test Accuracy = {v:.4f}")


Running for MAX_LEN = 80
Epoch 1 | Train Acc: 0.5083
Epoch 2 | Train Acc: 0.5364
Epoch 3 | Train Acc: 0.5559
Epoch 4 | Train Acc: 0.6056
Epoch 5 | Train Acc: 0.6360
Test Accuracy: 0.5723

Running for MAX_LEN = 100
Epoch 1 | Train Acc: 0.5115
Epoch 2 | Train Acc: 0.5451
Epoch 3 | Train Acc: 0.5449
Epoch 4 | Train Acc: 0.5672
Epoch 5 | Train Acc: 0.5724
Test Accuracy: 0.5293

Running for MAX_LEN = 120
Epoch 1 | Train Acc: 0.5184
Epoch 2 | Train Acc: 0.5464
Epoch 3 | Train Acc: 0.5577
Epoch 4 | Train Acc: 0.5760
Epoch 5 | Train Acc: 0.5890
Test Accuracy: 0.5805

Final Comparison:
MAX_LEN = 80 → Test Accuracy = 0.5723
MAX_LEN = 100 → Test Accuracy = 0.5293
MAX_LEN = 120 → Test Accuracy = 0.5805


In [ ]:
model, acc = train_and_evaluate(100)
print("Final Test Accuracy:", acc)


Running for MAX_LEN = 100
Epoch 1 | Train Acc: 0.5105
Epoch 2 | Train Acc: 0.5256
Epoch 3 | Train Acc: 0.5526
Epoch 4 | Train Acc: 0.5844
Epoch 5 | Train Acc: 0.6674
Test Accuracy: 0.7017
Final Test Accuracy: 0.70172


In [ ]:
def predict_sentiment(sentence, model, max_len):

    model.eval()

    encoded = encode(sentence, max_len)
    tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        output = model(tensor)
        probability = torch.sigmoid(output).item()

    sentiment = "Positive" if probability > 0.5 else "Negative"

    print("\nSentence:", sentence)
    print("Probability:", round(probability, 4))
    print("Predicted Sentiment:", sentiment)

In [ ]:
predict_sentiment("This movie was absolutely fantastic and amazing", model, 100)

predict_sentiment("The film was boring and I hated it", model, 100)

predict_sentiment("It was okay, not great but not terrible", model, 100)


Sentence: This movie was absolutely fantastic and amazing
Probability: 0.7262
Predicted Sentiment: Positive

Sentence: The film was boring and I hated it
Probability: 0.1853
Predicted Sentiment: Negative

Sentence: It was okay, not great but not terrible
Probability: 0.1721
Predicted Sentiment: Negative
